# Auto Shorts Maker (free)
Turns a long video into captioned 9:16 clips: **Whisper** transcribes, **Gemini** picks the best moments, **ffmpeg** cuts, crops and burns in word-highlight captions.

**One-time setup**
1. Runtime → Change runtime type → **T4 GPU** (faster transcription).
2. Get a free key at https://aistudio.google.com/apikey. In Colab click the 🔑 icon (left bar), add a secret named `GEMINI_API_KEY`, and turn on notebook access. (Or paste it into the settings cell.)

**Each time:** edit the settings cell, then Runtime → **Run all**. Only use videos you own or have permission to use.

In [ ]:
#@title 1. One-time Setup
# Force-install and upgrade critical libraries to prevent download failures
!pip install -q --upgrade av faster-whisper yt-dlp google-genai
!apt-get -qq install -y fonts-dejavu-core > /dev/null

import yt_dlp
print("✔ yt-dlp successfully upgraded and verified!")

In [ ]:
#@title Force Install Dependencies
!pip install -q --no-cache-dir av faster-whisper yt-dlp google-genai
import av
print("✔ 'av' library is now successfully installed and loaded!")

In [ ]:
# Redundant restart cell disabled to prevent infinite crash loop
print("Active library configuration verified.")

In [ ]:
import json
import re
import subprocess

def get_youtube_transcript_fast(youtube_url):
    """Attempts to download auto-generated or manual subtitles using yt-dlp to bypass Whisper transcription."""
    if not youtube_url.strip():
        return None, None

    print("⚡ Attempting to download existing transcript/subtitles directly from YouTube...", flush=True)
    import tempfile
    from pathlib import Path

    with tempfile.TemporaryDirectory() as tmpdir:
        # Download auto-generated or manual subtitles in WebVTT format
        cmd = [
            "python3", "-m", "yt_dlp",
            "--write-auto-subs", "--write-subs",
            "--sub-langs", "en",
            "--skip-download",
            "-o", f"{tmpdir}/sub",
            youtube_url.strip()
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

        # Find downloaded .vtt files
        vtt_files = list(Path(tmpdir).glob("*.vtt"))
        if not vtt_files:
            print("⚠ No pre-existing English subtitles found on YouTube. Falling back to local Whisper.", flush=True)
            return None, None

        vtt_path = vtt_files[0]
        print(f"✔ Downloaded YouTube subtitles: {vtt_path.name}", flush=True)
        return parse_vtt_to_whisper_format(vtt_path.read_text(encoding="utf-8"))

def parse_vtt_to_whisper_format(vtt_content):
    """Parses WebVTT content into Whisper-like segment and word dictionaries."""
    def parse_timestamp(ts_str):
        # Convert HH:MM:SS.fff or MM:SS.fff to total seconds
        parts = ts_str.strip().split(':')
        if len(parts) == 2:
            parts.insert(0, "00")
        h, m, s = parts
        return float(h) * 3600 + float(m) * 60 + float(s)

    segments = []
    words = []

    # Normalize line endings and split blocks
    blocks = re.split(r'\n\s*\n', vtt_content.strip())
    for block in blocks:
        lines = [l.strip() for l in block.split('\n') if l.strip()]
        if not lines or "WEBVTT" in lines[0] or "Kind:" in lines[0] or "Language:" in lines[0]:
            continue

        # Parse timecode line: e.g. 00:01:23.456 --> 00:01:26.789 align:start position:0%
        match = re.search(r'(\d+:\d+:\d+\.\d+|\d+:\d+\.\d+)\s*-->\s*(\d+:\d+:\d+\.\d+|\d+:\d+\.\d+)', lines[0])
        if not match:
            continue

        start_sec = parse_timestamp(match.group(1))
        end_sec = parse_timestamp(match.group(2))

        text_lines = lines[1:]
        # Strip WebVTT formatting tags from subtitles
        text = " ".join(text_lines)
        text = re.sub(r'<[^>]*>', '', text).strip()
        if not text:
            continue

        segments.append({
            "start": start_sec,
            "end": end_sec,
            "text": text
        })

        # Approximate word timestamps by dividing segment time equally
        split_words = text.split()
        if split_words:
            word_duration = (end_sec - start_sec) / len(split_words)
            for i, w in enumerate(split_words):
                words.append({
                    "start": start_sec + (i * word_duration),
                    "end": start_sec + ((i + 1) * word_duration),
                    "word": w
                })

    return segments, words

In [ ]:
#@title Settings
YOUTUBE_URL = ""          #@param {type:"string"}
VIDEO_PATH  = ""          #@param {type:"string"}

# Force globals assignment for safety
globals()['YOUTUBE_URL'] = YOUTUBE_URL
globals()['VIDEO_PATH'] = VIDEO_PATH

# MANUAL CLIP TIMES (Since we are skipping automatic transcription)
START_TIME = "00:00:00"   #@param {type:"string"}
END_TIME = ""     #@param {type:"string"}

LAYOUT      = "blur"      #@param ["blur", "crop"]
DISABLE_CAPTIONS = True   #@param {type:"boolean"}
CAPTION_WORDS_PER_LINE = 3   #@param {type:"integer"}
UPPERCASE_CAPTIONS = True    #@param {type:"boolean"}
SAVE_TO_DRIVE = True     #@param {type:"boolean"}
FONT = "DejaVu Sans"

In [ ]:
import os, re, json, subprocess, shutil
from pathlib import Path
import yt_dlp

def sh(cmd, cwd=None, stream_output=False):
    if stream_output:
        print(f"-> Running command: {' '.join(cmd[:8])}...", flush=True)
        process = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        while True:
            output = process.stdout.readline()
            if output == '' and process.poll() is not None:
                break
            if output:
                line = output.strip()
                if "frame=" in line or "fps=" in line or "size=" in line or "Error" in line:
                    print(f"   [ffmpeg] {line}", end="\r", flush=True)
        print() # New line after finishing progress stream
        rc = process.poll()
        if rc != 0:
            raise RuntimeError(f"Command failed with return code {rc}")
        return None
    else:
        r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("Command failed: " + " ".join(map(str, cmd[:3])) + "\n" + r.stderr[-1500:])
        return r

def get_video(url, path, workdir):
    """Return a local video path: download from URL natively with yt_dlp, use a given path, or ask for an upload."""
    if url.strip():
        # Clean up any partial files
        for p in workdir.glob("source.*"):
            try: p.unlink()
            except: pass

        # Bypassing the YouTube signature 'n-challenge' blocker by verifying Node.js is installed
        if not shutil.which("node"):
            print("⚡ Node.js not detected! Installing Node.js to solve YouTube challenge signatures...", flush=True)
            try:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "nodejs"], capture_output=True)
                print("✔ Node.js successfully installed!", flush=True)
            except Exception as node_err:
                print(f"⚠ System installation of Node.js failed: {node_err}. Proceeding anyway...", flush=True)

        print("Trying to download video natively using updated yt_dlp API...", flush=True)

        # Prepare robust extraction options using recommended YouTube client workarounds
        ydl_opts = {
            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
            'outtmpl': str(workdir / 'source.%(ext)s'),
            'merge_output_format': 'mp4',
            'noplaylist': True,
            'quiet': False,
            'nocheckcertificate': True,
            'ignoreerrors': False,
            'extractor_args': {
                'youtube': {
                    'player_client': ['android', 'webpage'],
                    'skip': ['dash', 'hls']
                }
            },
            'headers': {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
                'Accept-Language': 'en-US,en;q=0.9',
                'Sec-Fetch-Mode': 'navigate'
            }
        }

        # Automatically load cookies if user uploaded cookies.txt to bypass sign-in blocks
        cookies_file = Path("/content/cookies.txt")
        if cookies_file.exists():
            print("✔ Found custom cookies.txt file! Injecting authentication parameters...", flush=True)
            ydl_opts['cookiefile'] = str(cookies_file)
        else:
            print("⚠ Notice: No '/content/cookies.txt' file found. If download fails, please upload your Netscape format cookies.", flush=True)

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url.strip()])
            print("✔ Download completed!", flush=True)
        except Exception as e:
            print(f"⚠️ Primary download failed ({e}). Attempting robust single-stream fallback...", flush=True)
            # Clean up potential leftovers
            for p in workdir.glob("source.*"):
                try: p.unlink()
                except: pass

            fallback_opts = {
                'format': 'best',
                'outtmpl': str(workdir / 'source.%(ext)s'),
                'noplaylist': True,
                'quiet': False,
                'nocheckcertificate': True,
                'extractor_args': {
                    'youtube': {
                        'player_client': ['android']
                    }
                },
                'headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
                }
            }
            if cookies_file.exists():
                fallback_opts['cookiefile'] = str(cookies_file)

            with yt_dlp.YoutubeDL(fallback_opts) as ydl:
                ydl.download([url.strip()])
            print("✔ Fallback download completed!", flush=True)

        # Ensure we return the correct merged/downloaded file path
        possible_files = list(workdir.glob("source.mp4"))
        if possible_files:
            return str(possible_files[0])
        else:
            any_source = list(workdir.glob("source.*"))
            if any_source:
                fallback_file = any_source[0]
                if fallback_file.suffix != '.mp4':
                    target = fallback_file.with_suffix('.mp4')
                    shutil.move(str(fallback_file), str(target))
                    return str(target)
                return str(fallback_file)
            raise RuntimeError("Could not find downloaded source video file. Check that your 'cookies.txt' formatting is correct.")

    if path.strip():
        return path.strip()
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    dest = workdir / name
    shutil.move(name, dest)
    return str(dest)

In [ ]:
# This manual clip rendering cell has been deactivated to keep your notebook clean and hands-free.
# Please proceed directly to Step 3 below for automated clip selection!
print("Manual cell deactivated. Proceed to Step 3 below.")

### 3. Automated Clip Selection and Rendering with Gemini
This cell transcribes your video, uses Gemini to select the most funny and self-contained moments between 40 and 90 seconds, and then renders them with automatic word-level captions.

In [ ]:
# @title 3. Automated Clip Selection and Rendering with Gemini
from google.colab import userdata
import os, sys, shutil, json
from pathlib import Path
import importlib
import av

# Cleanly reload av
importlib.reload(av)
_genuine_av_open = av.open
while hasattr(_genuine_av_open, "__wrapped__"):
    _genuine_av_open = _genuine_av_open.__wrapped__

def _clean_patched_av_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return _genuine_av_open(*args, **kwargs)

av.open = _clean_patched_av_open

try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = ""

if not api_key:
    raise ValueError("Please set your GEMINI_API_KEY in the Colab Secrets panel (🔑).")

yt_url = globals().get('YOUTUBE_URL', "").strip()
vid_path = globals().get('VIDEO_PATH', "").strip()
workdir = Path("/content/work"); workdir.mkdir(exist_ok=True)
outdir = Path("/content/shorts")

transcribe_cache_file = workdir / "transcription_cache.json"
gemini_cache_file = workdir / "gemini_clips_cache.json"
metadata_file = workdir / "video_metadata.json"

# Detect video source change to invalidate old cache files automatically
current_source = yt_url if yt_url else vid_path
previous_source = ""
if metadata_file.exists():
    try:
        with open(metadata_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
            previous_source = meta.get("source", "")
    except Exception:
        pass

if current_source != previous_source:
    print("🔄 New video source detected! Cleaning up old cache and temporary files...", flush=True)
    if transcribe_cache_file.exists():
        transcribe_cache_file.unlink()
    if gemini_cache_file.exists():
        gemini_cache_file.unlink()
    if outdir.exists():
        shutil.rmtree(outdir)
    if (workdir / "source.mp4").exists():
        (workdir / "source.mp4").unlink()
    # Reset global execution states
    globals().pop('segments', None)
    globals().pop('words', None)
    globals().pop('video', None)

    # Save the updated source to the metadata file
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump({"source": current_source}, f, indent=2)

# Ensure output directory exists
outdir.mkdir(exist_ok=True)

# Retrieve video file
video_file = globals().get('video', None)
if not video_file or not Path(video_file).exists():
    print("Source video not loaded in memory. Retrieving video file...", flush=True)
    video_file = get_video(yt_url, vid_path, workdir)
    globals()['video'] = video_file

# Custom run whisper function with step-by-step progress logging
def _run_whisper_logged(video_path, size, device, ctype, language):
    from faster_whisper import WhisperModel
    model = WhisperModel(size, device=device, compute_type=ctype)
    segs, info = model.transcribe(video_path, word_timestamps=True, vad_filter=True, language=language)

    total_duration = info.duration
    print(f"-> Audio duration: {total_duration:.1f} seconds. Starting transcription streaming...", flush=True)

    segments, words = [], []
    last_percent_print = -5

    for s in segs:
        segments.append({"start": s.start, "end": s.end, "text": s.text.strip()})
        for w in (s.words or []):
            words.append({"start": w.start, "end": w.end, "word": w.word.strip()})

        if total_duration > 0:
            pct = int((s.end / total_duration) * 100)
            if pct >= last_percent_print + 5 or pct >= 100:
                print(f"   Transcribing progress: {pct}% complete (at video timestamp: {s.end:.1f}s)...", flush=True)
                last_percent_print = pct

    return segments, words

def transcribe_logged(video_path, size="auto", language=None):
    has_gpu = shutil.which("nvidia-smi") is not None
    if has_gpu:
        try:
            return _run_whisper_logged(video_path, "medium" if size == "auto" else size, "cuda", "float16", language)
        except Exception as e:
            print("GPU transcription failed, falling back to CPU:", str(e)[:200], flush=True)
    return _run_whisper_logged(video_path, "small" if size == "auto" else size, "cpu", "int8", language)

# --- STEP 1: TRANSCRIPTION (with local file-cache validation) ---
segments, words = None, None
if transcribe_cache_file.exists():
    try:
        with open(transcribe_cache_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
            # Ensure loaded data is valid and non-empty
            if cache_data and isinstance(cache_data.get("segments"), list) and isinstance(cache_data.get("words"), list):
                print("✔ Found valid local transcription cache. Loading elements to skip Step 1...", flush=True)
                segments = cache_data.get("segments")
                words = cache_data.get("words")
                globals()['segments'] = segments
                globals()['words'] = words
            else:
                print("⚠ Found transcription cache but data was empty or invalid. Re-transcribing...", flush=True)
    except Exception as e:
        print(f"⚠ Local cache read failed ({e}). Proceeding with standard transcription.", flush=True)

if not segments or not words:
    # Fallback to active memory if global variables are present and valid
    if 'segments' in globals() and isinstance(globals()['segments'], list) and 'words' in globals() and isinstance(globals()['words'], list):
        print("✔ Reusing existing completed transcription from active memory!", flush=True)
        segments = globals()['segments']
        words = globals()['words']
    else:
        print("1/4 Transcribing video with Whisper... (this may take a few minutes)", flush=True)
        try:
            segments, words = transcribe_logged(video_file, size="auto")
            globals()['segments'] = segments
            globals()['words'] = words
            with open(transcribe_cache_file, "w", encoding="utf-8") as f:
                json.dump({"segments": segments, "words": words}, f, indent=2)
            print("✔ Transcription completed and saved to local disk cache!", flush=True)
        except Exception as e:
            print(f"⚠ Whisper transcription failed: {e}", flush=True)
            raise e

# --- STEP 2: GEMINI AI SELECTION (with local file-cache validation) ---
raw_clips = None
if gemini_cache_file.exists():
    try:
        with open(gemini_cache_file, "r", encoding="utf-8") as f:
            loaded_clips = json.load(f)
            if loaded_clips and isinstance(loaded_clips, list):
                print("✔ Found local Gemini clips cache. Loading elements to skip Step 2...", flush=True)
                raw_clips = loaded_clips
            else:
                print("⚠ Found Gemini cache but data was invalid. Re-querying AI...", flush=True)
    except Exception as e:
        print(f"⚠ Local Gemini cache read failed ({e}). Re-querying.", flush=True)

if not raw_clips:
    print("2/4 Finding entertaining clips using Gemini...", flush=True)
    try:
        # Set a minimum limit of 5 clips, with no maximum limit (letting Gemini choose)
        raw_clips = pick_clips(
            segments=segments,
            api_key=api_key,
            model="gemini-3.8-flash",
            n=5,
            min_s=40.0,
            max_s=90.0
        )
        with open(gemini_cache_file, "w", encoding="utf-8") as f:
            json.dump(raw_clips, f, indent=2)
        print(f"Gemini selected {len(raw_clips)} funny/entertaining clips and cached them!", flush=True)
    except Exception as e:
        print(f"⚠ Gemini selection failed: {e}", flush=True)
        raise e

# --- STEP 3: BOUNDARY REFINEMENT ---
print("3/4 Aligning and refining clip boundaries...", flush=True)
# Pass n=None to keep all candidates selected by Gemini
clips = clean_clips(raw_clips, segments, n=None, min_s=40.0, max_s=90.0)
print(f"Refined into {len(clips)} valid, non-overlapping clips.", flush=True)

# --- STEP 4: RENDERING (with target video existence check) ---
print("4/4 Rendering clips...", flush=True)
notes = []
for idx, clip in enumerate(clips, 1):
    expected_output_name = f"{idx:02d}_{slug(clip['title'])}.mp4"
    expected_output_path = outdir / expected_output_name

    if expected_output_path.exists():
        print(f"✔ Clip #{idx} ({expected_output_name}) already exists! Skipping render.", flush=True)
        notes.append(f"{expected_output_name}\nTitle: {clip['title']}\nHook: {clip['hook']}\n")
        continue

    print(f"Rendering Clip #{idx}: \"{clip['title']}\" ({clip['start']:.1f}s to {clip['end']:.1f}s)", flush=True)
    try:
        path = render_clip(
            video=video_file,
            clip=clip,
            words=words,
            outdir=outdir,
            idx=idx,
            mode=globals().get('LAYOUT', 'blur'),
            workdir=workdir,
            font=globals().get('FONT', 'DejaVu Sans'),
            per_line=globals().get('CAPTION_WORDS_PER_LINE', 3),
            upper=globals().get('UPPERCASE_CAPTIONS', True),
            disable_captions=globals().get('DISABLE_CAPTIONS', False)
        )
    except Exception as render_err:
        print(f"  Subtitled render failed ({render_err}), rendering fallback clip without captions...", flush=True)
        path = render_clip(
            video=video_file,
            clip=clip,
            words=[],
            outdir=outdir,
            idx=idx,
            mode=globals().get('LAYOUT', 'blur'),
            workdir=workdir,
            font=globals().get('FONT', 'DejaVu Sans'),
            per_line=globals().get('CAPTION_WORDS_PER_LINE', 3),
            upper=globals().get('UPPERCASE_CAPTIONS', True),
            disable_captions=True
        )
    notes.append(f"{Path(path).name}\nTitle: {clip['title']}\nHook: {clip['hook']}\n")

(outdir / "titles.txt").write_text("\n".join(notes), encoding="utf-8")
print("✔ All automated clips are fully generated! Proceed to the next cell to download them!", flush=True)

In [ ]:
# @title 3. Automated Clip Selection and Rendering with Gemini (Optimized with Fast YouTube Transcript Fetch)
from google.colab import userdata
import os, sys, shutil, json, re
from pathlib import Path
import importlib
import av

# Cleanly reload av
importlib.reload(av)
_genuine_av_open = av.open
while hasattr(_genuine_av_open, "__wrapped__"):
    _genuine_av_open = _genuine_av_open.__wrapped__

def _clean_patched_av_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return _genuine_av_open(*args, **kwargs)

av.open = _clean_patched_av_open

try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = ""

if not api_key:
    raise ValueError("Please set your GEMINI_API_KEY in the Colab Secrets panel (🔑).")

yt_url = globals().get('YOUTUBE_URL', "").strip()
vid_path = globals().get('VIDEO_PATH', "").strip()

workdir = Path("/content/work"); workdir.mkdir(exist_ok=True)
outdir = Path("/content/shorts")

transcribe_cache_file = workdir / "transcription_cache.json"
gemini_cache_file = workdir / "gemini_clips_cache.json"
metadata_file = workdir / "video_metadata.json"

current_source = yt_url if yt_url else vid_path
previous_source = ""
if metadata_file.exists():
    try:
        with open(metadata_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
            if isinstance(meta, dict):
                previous_source = meta.get("source", "")
    except Exception:
        pass

# If the URL has changed, or if there's no valid previous source, do an absolute hard clean!
if current_source != previous_source or not previous_source:
    print(f"🔄 URL change detected! Resetting pipeline for fresh source: {current_source}", flush=True)

    # Clear out all files from the workspace
    if transcribe_cache_file.exists():
        try: transcribe_cache_file.unlink()
        except: pass
    if gemini_cache_file.exists():
        try: gemini_cache_file.unlink()
        except: pass
    if outdir.exists():
        try: shutil.rmtree(outdir)
        except: pass
    outdir.mkdir(exist_ok=True)

    for p in workdir.glob("source.*"):
        try: p.unlink()
        except: pass
    for p in workdir.glob("*.ass"):
        try: p.unlink()
        except: pass

    # Forcefully pop previous video data out of Python memory
    globals().pop('segments', None)
    globals().pop('words', None)
    globals().pop('video', None)
    globals().pop('raw_clips', None)
    globals().pop('clips', None)
    globals().pop('notes', None)

    # Write down the new source to prevent infinite resets
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump({"source": current_source}, f, indent=2)

# Ensure output directory exists
outdir.mkdir(exist_ok=True)

# Retrieve video file
video_file = globals().get('video', None)
if not video_file or not Path(video_file).exists():
    print(f"Retrieving video file for: {yt_url if yt_url else vid_path}...", flush=True)
    try:
        video_file = get_video(yt_url, vid_path, workdir)
        globals()['video'] = video_file
    except Exception as e:
        print("❌ Download failed! Wiping state files so next attempt is fully clean.", flush=True)
        if metadata_file.exists():
            try: metadata_file.unlink()
            except: pass
        globals().pop('video', None)
        raise e

# Custom run whisper function with step-by-step progress logging
def _run_whisper_logged(video_path, size, device, ctype, language):
    from faster_whisper import WhisperModel
    model = WhisperModel(size, device=device, compute_type=ctype)
    segs, info = model.transcribe(video_path, word_timestamps=True, vad_filter=True, language=language)

    total_duration = info.duration
    print(f"-> Audio duration: {total_duration:.1f} seconds. Starting transcription streaming...", flush=True)

    segments, words = [], []
    last_percent_print = -5

    for s in segs:
        segments.append({"start": s.start, "end": s.end, "text": s.text.strip()})
        for w in (s.words or []):
            words.append({"start": w.start, "end": w.end, "word": w.word.strip()})

        if total_duration > 0:
            pct = int((s.end / total_duration) * 100)
            if pct >= last_percent_print + 5 or pct >= 100:
                print(f"   Transcribing progress: {pct}% complete (at video timestamp: {s.end:.1f}s)...", flush=True)
                last_percent_print = pct

    return segments, words

def transcribe_logged(video_path, size="auto", language=None):
    has_gpu = shutil.which("nvidia-smi") is not None
    if has_gpu:
        try:
            return _run_whisper_logged(video_path, "medium" if size == "auto" else size, "cuda", "float16", language)
        except Exception as e:
            print("GPU transcription failed, falling back to CPU:", str(e)[:200], flush=True)
    return _run_whisper_logged(video_path, "small" if size == "auto" else size, "cpu", "int8", language)

# --- STEP 1: TRANSCRIPTION ---
segments, words = None, None
if transcribe_cache_file.exists():
    try:
        with open(transcribe_cache_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
            if cache_data and isinstance(cache_data.get("segments"), list) and isinstance(cache_data.get("words"), list):
                print("✔ Found valid local transcription cache. Loading elements to skip Step 1...", flush=True)
                segments = cache_data.get("segments")
                words = cache_data.get("words")
                globals()['segments'] = segments
                globals()['words'] = words
    except Exception as e:
         print(f"⚠ Local cache read failed ({e}). Proceeding.", flush=True)

if not segments or not words:
    if 'segments' in globals() and isinstance(globals()['segments'], list) and 'words' in globals() and isinstance(globals()['words'], list):
        print("✔ Reusing existing completed transcription from active memory!", flush=True)
        segments = globals()['segments']
        words = globals()['words']
    else:
        # Try Fast YouTube Subtitles method first
        if yt_url:
            try:
                segments, words = get_youtube_transcript_fast(yt_url)
            except Exception as e:
                print(f"⚠ Could not fetch YouTube captions online: {e}", flush=True)

        # Fallback to local Whisper if YouTube retrieval fails or is not a YouTube URL
        if not segments or not words:
            print("1/4 Transcribing video with Whisper... (this may take a few minutes)", flush=True)
            try:
                segments, words = transcribe_logged(video_file, size="auto")
            except Exception as e:
                print(f"⚠ Whisper transcription failed: {e}", flush=True)
                raise e

        # Store the retrieved or transcribed subtitles in the local cache
        globals()['segments'] = segments
        globals()['words'] = words
        with open(transcribe_cache_file, "w", encoding="utf-8") as f:
            json.dump({"segments": segments, "words": words}, f, indent=2)
        print("✔ Transcription complete and saved!", flush=True)

# --- STEP 2: GEMINI AI SELECTION ---
raw_clips = None
if gemini_cache_file.exists():
    try:
        with open(gemini_cache_file, "r", encoding="utf-8") as f:
            loaded_clips = json.load(f)
            if loaded_clips and isinstance(loaded_clips, list):
                print(f"✔ Found local Gemini clips cache. Loaded {len(loaded_clips)} clips to skip Step 2...", flush=True)
                raw_clips = loaded_clips
    except Exception as e:
        pass

if not raw_clips:
    print("2/4 Finding entertaining clips using Gemini...", flush=True)
    try:
        raw_clips = pick_clips(
            segments=segments,
            api_key=api_key,
            model="gemini-3.8-flash",
            n=5,
            min_s=40.0,
            max_s=90.0
        )
        with open(gemini_cache_file, "w", encoding="utf-8") as f:
            json.dump(raw_clips, f, indent=2)
        print(f"✔ Gemini successfully selected {len(raw_clips)} candidate clips and cached them!", flush=True)
    except Exception as e:
        print(f"⚠ Gemini selection failed: {e}", flush=True)
        raise e

# --- STEP 3: BOUNDARY REFINEMENT ---
print("3/4 Aligning and refining clip boundaries...", flush=True)
clips = clean_clips(raw_clips, segments, n=None, min_s=40.0, max_s=90.0)
print(f"✔ Refined into {len(clips)} valid, non-overlapping clips.", flush=True)

# --- STEP 4: RENDERING ---
print(f"4/4 Rendering {len(clips)} total clips...", flush=True)
notes = []
for idx, clip in enumerate(clips, 1):
    expected_output_name = f"{idx:02d}_{slug(clip['title'])}.mp4"
    expected_output_path = outdir / expected_output_name

    if expected_output_path.exists():
        print(f"✔ Clip #{idx}/{len(clips)} ({expected_output_name}) already exists! Skipping render.", flush=True)
        notes.append(f"{expected_output_name}\nTitle: {clip['title']}\nHook: {clip['hook']}\n")
        continue

    print(f"[Progress {idx}/{len(clips)}] Rendering: \"{clip['title']}\" ({clip['start']:.1f}s to {clip['end']:.1f}s)...", flush=True)
    try:
        path = render_clip(
            video=video_file,
            clip=clip,
            words=words,
            outdir=outdir,
            idx=idx,
            mode=globals().get('LAYOUT', 'blur'),
            workdir=workdir,
            font=globals().get('FONT', 'DejaVu Sans'),
            per_line=globals().get('CAPTION_WORDS_PER_LINE', 3),
            upper=globals().get('UPPERCASE_CAPTIONS', True),
            disable_captions=globals().get('DISABLE_CAPTIONS', False)
        )
    except Exception as render_err:
        print(f"  ⚠️ Subtitled render failed ({render_err}), rendering fallback clip without captions...", flush=True)
        path = render_clip(
            video=video_file,
            clip=clip,
            words=[],
            outdir=outdir,
            idx=idx,
            mode=globals().get('LAYOUT', 'blur'),
            workdir=workdir,
            font=globals().get('FONT', 'DejaVu Sans'),
            per_line=globals().get('CAPTION_WORDS_PER_LINE', 3),
            upper=globals().get('UPPERCASE_CAPTIONS', True),
            disable_captions=True
        )
    print(f"✔ Successfully finished rendering clip #{idx}!\n", flush=True)
    notes.append(f"{Path(path).name}\nTitle: {clip['title']}\nHook: {clip['hook']}\n")

(outdir / "titles.txt").write_text("\n".join(notes), encoding="utf-8")
print("✔ All automated clips are fully generated! Proceed to the next cell to download them!", flush=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

# Fallbacks if previous cells were not fully loaded in memory
save_to_drive_enabled = globals().get('SAVE_TO_DRIVE', False)
outdir = Path("/content/shorts")

# Ensure the output directory physically exists
outdir.mkdir(parents=True, exist_ok=True)

# List files in the shorts directory
existing_files = list(outdir.glob("*"))
print(f"Files ready for download in {outdir}: {[f.name for f in existing_files]}")

if not existing_files:
    print("⚠ No generated files found in the shorts directory! Please make sure you ran the manual rendering cell (nWXPUzglJUxX) first.")
else:
    if save_to_drive_enabled:
        try:
            from google.colab import drive
            print("Attempting to mount Google Drive...")
            drive.mount("/content/drive")
            shutil.copytree(outdir, "/content/drive/MyDrive/shorts", dirs_exist_ok=True)
            print("✔ Saved successfully to Google Drive > shorts")
        except Exception as e:
            print(f"⚠ Google Drive mount failed: {e}")
            print("Skipping Google Drive save, proceeding to download the zip file directly.")

    # Create a ZIP archive of your short(s)
    print("Archiving shorts...")
    shutil.make_archive("/content/shorts", "zip", outdir)

    # Download the file to your computer
    print("Downloading shorts.zip...")
    files.download("/content/shorts.zip")